# 01 - Synthetic Data Generation

Generates the RiskShield AI transaction dataset used throughout the rest of the pipeline. See `data/generate_data.py` for the full generator; this notebook runs it and inspects the output. Six fraud scenarios are embedded deliberately (not random label flips) so the ML models have learnable structure to find.

In [1]:
import subprocess, sys
sys.path.insert(0, '.')
result = subprocess.run([sys.executable, 'data/generate_data.py'], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

Generating base entities...
Generating 92000 legitimate transactions...
Injecting Scenario A: velocity attacks...
Injecting Scenario B: account takeover...
Injecting Scenario C: card testing...
Injecting Scenario D/E: device sharing & coordinated abuse rings...
Injecting Scenario F: geographic anomalies...
Assembling final dataset...
Done. 97,424 transactions generated (5,274 fraud / 5.41% fraud rate)
scenario
abuse_ring           1080
account_takeover      515
card_testing         1581
geo_anomaly           150
legit               92150
velocity_attack      1948
dtype: int64




## Inspect the generated files

In [2]:
import pandas as pd
txns = pd.read_csv('data/processed/transactions.csv')
scenario_labels = pd.read_csv('data/processed/transactions_scenario_labels.csv')
txns = txns.merge(scenario_labels, on='transaction_id')
print(f'{len(txns):,} transactions')
print(f'Fraud rate: {txns.label.mean()*100:.2f}%')
txns.groupby('scenario').size().sort_values(ascending=False)

97,424 transactions
Fraud rate: 5.41%


scenario
legit               92150
velocity_attack      1948
card_testing         1581
abuse_ring           1080
account_takeover      515
geo_anomaly           150
dtype: int64

In [3]:
customers = pd.read_csv('data/processed/customers.csv')
devices = pd.read_csv('data/processed/devices.csv')
ips = pd.read_csv('data/processed/ips.csv')
merchants = pd.read_csv('data/processed/merchants.csv')
print(f'Customers: {len(customers):,}  Devices: {len(devices):,}  IPs: {len(ips):,}  Merchants: {len(merchants):,}')

Customers: 8,000  Devices: 8,516  IPs: 8,491  Merchants: 40


## Sanity check: fraud scenarios look distinguishable from legit traffic

In [4]:
txns.groupby('scenario')['amount'].describe()[['mean', 'std', 'min', 'max']]

,mean,std,min,max
scenario,,,,
abuse_ring,2168.944463,1094.145229,311.28,3996.03
account_takeover,6153.518680,4718.382898,578.12,31507.22
card_testing,660.477445,1760.749437,5.02,7960.41
geo_anomaly,2341.360067,2014.457840,281.16,11809.07
legit,627.948006,535.509181,20.00,9803.10
velocity_attack,771.107582,598.553610,84.27,5293.33
